In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [ ]:
df1 = pd.read_csv("./combined_features/resnet50_data.csv")
df2 = pd.read_csv("./combined_features/vitbase_data.csv")
print(len(df1))
print(len(df2))

In [ ]:
def get_features_extractors(folder_path):
    extractor = []
    for file in glob.glob(os.path.join(folder_path, "*.csv")):
        file_name = Path(file).stem
        feature_extractor = file_name.split("_")[0]
        extractor.append(feature_extractor)
        
    return list(set(extractor))


In [ ]:
def gather_data(folder_path):
    extractors = get_features_extractors(folder_path)
    dataframes = {}
    os.makedirs("./combined_features", exist_ok=True)

    for extractor in extractors:
        dfs = []
        for file in glob.glob(os.path.join(folder_path, f"{extractor}*.csv")):
            df = pd.read_csv(file)
            dfs.append(df)
        
        if dfs:
            combined = pd.concat(dfs, ignore_index=True)
            combined.to_csv(f"./combined_features/{extractor}_data.csv", index=False)
            



In [ ]:
def clean_normalize_data(data_path, metadata_path):
    metadata_df = pd.read_csv(metadata_path)
    metadata_df = metadata_df[["isic_id", "patient_id", "target"]]
    df_list = {}
    for csv_path in glob.glob(os.path.join(data_path, "*.csv")):
        file_name = Path(csv_path).stem
        if file_name == "metadata":
            df = pd.read_csv(csv_path)
            df.drop("skin_type", axis=1, inplace=True)
            feature_extractor = file_name.split("_")[0]
            df_list[feature_extractor] = df
            continue
        feature_extractor = file_name.split("_")[0]
        
        df = pd.read_csv(csv_path)
        df["isic_id"] = df["filename"].str.split(".").str[0]
        df = pd.merge(df, metadata_df, on="isic_id")
        df.drop("filename", axis=1, inplace=True)
        df.sort_values("isic_id", inplace=True)
        df.reset_index(drop=True, inplace=True)
        df_list[feature_extractor] = df
    return df_list    

In [ ]:
def run_crossval_generic(
    model,                
    df_dict,
    model_pipeline,       
    param_grid,           
    save_dir="results",
    target_col="target",
    group_col="patient_id",
    id_col="isic_id",
    n_splits_outer=10,
    n_splits_inner=3,
    random_state=100):
    
    os.makedirs(save_dir, exist_ok=True)
    results = {}

    for fe_name, df in df_dict.items():
        print(f"\n🔹 Running {model} for Feature Extractor: {fe_name}")
        
        # ⚠️ ALERTA DE DEBUG: Imprimindo o tamanho para você ver a diferença
        print(f"   Shape do dataset: {df.shape}")

        X = df.drop([target_col, group_col, id_col], axis=1).values
        y = df[target_col].values
        ids = df[id_col].values
        groups = df[group_col].values

        # 💡 CORREÇÃO AQUI: Os folds agora são gerados para CADA DataFrame individualmente
        cv_outer = StratifiedGroupKFold(n_splits=n_splits_outer, shuffle=True, random_state=random_state)
        folds = list(cv_outer.split(df, y, groups))

        preds_list = []
        best_params_list = []

        # O resto do loop continua idêntico...
        for fold_idx, (train_idx, test_idx) in enumerate(folds):
            print(f"  Fold {fold_idx+1}/{n_splits_outer}")

            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            ids_test = ids[test_idx]
            groups_test = groups[test_idx]
            groups_train = groups[train_idx] 

            cv_inner = StratifiedGroupKFold(n_splits=n_splits_inner, shuffle=True, random_state=random_state)

            grid = GridSearchCV(
                estimator=model_pipeline, 
                param_grid=param_grid,    
                cv=cv_inner,
                scoring="f1_macro",
                n_jobs=-1
            )
            
            grid.fit(X_train, y_train, groups=groups_train)

            best_model = grid.best_estimator_
            best_params_list.append(grid.best_params_)

            y_proba = best_model.predict_proba(X_test)
            y_pred = best_model.predict(X_test)

            fold_df = pd.DataFrame({
                id_col: ids_test,
                group_col: groups_test,
                "fold": fold_idx,
                "true_label": y_test,
                "prob_class_0": y_proba[:, 0],
                "prob_class_1": y_proba[:, 1],
                "pred_label": y_pred
            })
            preds_list.append(fold_df)

        preds_df = pd.concat(preds_list, ignore_index=True)

        acc = accuracy_score(preds_df["true_label"], preds_df["pred_label"])
        f1 = f1_score(preds_df["true_label"], preds_df["pred_label"], average="macro")
        f1_weighted = f1_score(preds_df["true_label"], preds_df["pred_label"], average="weighted")
        prec_macro = precision_score(preds_df["true_label"], preds_df["pred_label"], average="macro", zero_division=0)
        rec_macro = recall_score(preds_df["true_label"], preds_df["pred_label"], average="macro", zero_division=0)

        print(f" ACC={acc:.3f}, F1-Macro={f1:.3f}, Precision={prec_macro:.3f}, Recall={rec_macro:.3f}")

        preds_df.to_csv(os.path.join(save_dir, f"{fe_name}_{model}_predictions.csv"), index=False)

        results[fe_name] = {
            "accuracy": acc,
            "f1_macro": f1,
            "f1_weighted": f1_weighted,
            "precision_macro": prec_macro,
            "recall_macro": rec_macro,
            "best_params": best_params_list
        }

    summary = pd.DataFrame(
        {fe: {k: v for k, v in vals.items() if k != "best_params"} for fe, vals in results.items()}
    ).T
    summary.to_csv(os.path.join(save_dir, f"{model}_summary_results.csv"))

    print(f"\n📊 Summary of {model} results:")
    print(summary)

    return results

In [ ]:
# gather_data("./features/")

In [ ]:
metadata_path = "./all_skin.csv"
data_path = "./combined_features/"
df_dicts = clean_normalize_data(data_path, metadata_path)

In [ ]:
df_dicts.keys()

In [ ]:
dt_pipeline = Pipeline([
    ("clf", DecisionTreeClassifier(random_state=100))
])

# 2. Grid de hiperparâmetros convertido para Python
dt_param_grid = {
    "clf__criterion": ['entropy'],
    "clf__splitter": ['best', 'random'],
    "clf__max_depth": [None, 10, 20, 30, 50],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__max_features": [None, 'sqrt', 'log2'],
    "clf__class_weight": [None, 'balanced']
}

# 3. Rodar a validação cruzada passando a string de identificação "DecisionTree"
resultados_dt = run_crossval_generic(
    model="DecisionTree",        # Identificador adicionado aqui
    df_dict=df_dicts,     # Substitua pelo seu dicionário de dataframes reais
    model_pipeline=dt_pipeline,
    param_grid=dt_param_grid,
    save_dir="results"           # Salvará arquivos como 'results/GLCM_DecisionTree_predictions.csv'
)

In [ ]:
df_dicts["vitbase"]

In [ ]:
#0.777, 0.676
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(probability=True, random_state=100))
])

# 2. Grid de hiperparâmetros (convertido para o formato do Pipeline)
svm_param_grid = {
    "clf__C": [0.1, 1, 10, 100, 1000],
    "clf__kernel": ['rbf'],
    "clf__gamma": ['scale', 'auto'],
    "clf__class_weight": [None, 'balanced']
}

# 3. Rodar a validação cruzada passando a string de identificação "SVM"
resultados_svm = run_crossval_generic(
    model="SVM",                 # Identificador para os logs e nomes dos arquivos
    df_dict=df_dicts,     # Substitua pelo seu dicionário de dataframes
    model_pipeline=svm_pipeline,
    param_grid=svm_param_grid,
    save_dir="results"           # Salvará arquivos como 'results/GLCM_SVM_predictions.csv'
)

In [ ]:
df_dicts.pop("vitbase")

In [ ]:
mlp_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(random_state=100))
])

# 2. Grid de hiperparâmetros (convertido e com prefixo clf__)
mlp_param_grid = {
    "clf__hidden_layer_sizes": [(50,), (100,), (50, 50), (100, 50)],
    "clf__activation": ['tanh', 'relu', 'logistic'],
    "clf__solver": ['adam'],
    "clf__alpha": [0.0001, 0.001, 0.01, 0.1],
    "clf__learning_rate": ['constant', 'adaptive'],
    "clf__max_iter": [100, 150, 200]
}

# 3. Rodar a validação cruzada passando a string de identificação "MLP"
resultados_mlp = run_crossval_generic(
    model="MLP",                 # Identificador para os logs e arquivos
    df_dict=df_dicts,     # Substitua pelo seu dicionário
    model_pipeline=mlp_pipeline,
    param_grid=mlp_param_grid,
    save_dir="results"           
)

In [ ]:
#vitbase 0.7848
#resnet50 0.6693